# TC_03: FVNN clean baseline (Golden Baseline)
Full diagnostic pipeline on clean data using RF model.
UQ: Deep Ensemble, MC Dropout
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report


config = load_config()
tracker = PerformanceTracker()

### Load data and EDA


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, "FCNN Diabetes Baseline", config)
plot_correlation(report, "FCNN Diabetes Baseline", config)
plot_feature_boxplots(X_train, "FCNN Diabetes Baseline", config)

Loading dataset.
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators
Saved: results/figures\class_distribution_fcnn_diabetes_baseline.png
Saved: results/figures\correlation_fcnn_diabetes_baseline.png
Saved: results/figures\boxplot_fcnn_diabetes_baseline.png


### Train FCNN baseline

In [3]:
tracker.start_performance_track()
fcnn_model = train_fcnn(X_train, y_train, config)
tracker.stop_performance_track("FCNN training")

tracker.start_performance_track()
fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
tracker.stop_performance_track("FCNN evaluation")

FCNN training started.
Epoch 10/50
 Early stopping at epoch 20
FCNN training completed.
FCNN training: Time:57.23s, Memory:184.32MB
Model evaluation started for FCNN.
{'0': {'precision': 0.877029544849614, 'recall': 0.9809467103304554, 'f1-score': 0.9260820685778527, 'support': 43667.0}, '1': {'precision': 0.5609498680738786, 'recall': 0.150374876220116, 'f1-score': 0.23717090584560463, 'support': 7069.0}, 'accuracy': 0.8652239041311889, 'macro avg': {'precision': 0.7189897064617463, 'recall': 0.5656607932752857, 'f1-score': 0.5816264872117287, 'support': 50736.0}, 'weighted avg': {'precision': 0.8329904555416735, 'recall': 0.8652239041311889, 'f1-score': 0.8300967128274139, 'support': 50736.0}}
FCNN evaluation: Time:0.02s, Memory:12.70MB


{'operation': 'FCNN evaluation', 'time_seconds': 0.02, 'memory_usage': 12.7}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [4]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
tracker.stop_performance_track("FCNN MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "FCNN MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for tabular.
FCNN MC Dropout: Time:0.08s, Memory:22.62MB
MC Dropout uncertainty status:
Mean: 0.025179646909236908
Max: 0.15113168954849243
 90th percentile (threshold): 0.0515
Saved: results/figures\uncertainty_distribution_fcnn_mc_dropout.png


In [5]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train, y_train, X_test, config)
tracker.stop_performance_track("FCNN Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "FCNN Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular.
Training model with seed 0
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
 Early stopping at epoch 39
FCNN training completed.
Training model with seed 1
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 2
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
Epoch 40/50
Epoch 50/50
FCNN training completed.
Training model with seed 3
FCNN training started.
Epoch 10/50
Epoch 20/50
Epoch 30/50
 Early stopping at epoch 34
FCNN training completed.
Training model with seed 4
FCNN training started.
Epoch 10/50
Epoch 20/50
 Early stopping at epoch 23
FCNN training completed.
Deep Ensemble finished for tabular.
FCNN Deep Ensemble prediction: Time:554.69s, Memory:0.00MB
Deep Ensembl uncertainty status:
Mean: 0.012324007228016853
Max: 0.1385054886341095
 90th percentile (threshold): 0.0302
Saved: results/figures\uncertainty_distribution_fcnn_deep_ensemble.png


### Explainability (SHAP and LIME)

In [6]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
tracker.stop_performance_track("FCNN SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP: Time:7.56s, Memory:0.00MB
SHAP values shape: (200, 21, 2)


In [7]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
tracker.stop_performance_track("FCNN LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME: Time:2.44s, Memory:14.27MB
LIME explanations: 200


In [8]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.565
Min consistency: 0.2
Max consistency: 1.0


### Flagging (MC Dropout + Consistency vs Deep Ensembles + Consistency)

In [9]:
# MC Dropout flags
mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)

print(f"MC Dropout + XAI Flagging:")
mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(mc_flags, "FCNN MC Dropout + XAI Flagging", config)

MC Dropout + XAI Flagging:
RED: Count: 4 Accuracy:75.0%
YELLOW: Count: 67 Accuracy:85.1%
GREEN: Count: 129 Accuracy:88.4%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_+_xai_flagging.png


In [10]:
# Deep Ensemble flags
de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)

print(f"Deep Ensembles + XAI Flagging:")
de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(de_flags, "FCNN Deep Ensembles + XAI Flagging", config)

Deep Ensembles + XAI Flagging:
RED: Count: 6 Accuracy:83.3%
YELLOW: Count: 68 Accuracy:80.9%
GREEN: Count: 126 Accuracy:90.5%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_+_xai_flagging.png


### UQ only vs UQ + XAI comparison

In [11]:
# MC Dropout without XAI (constant consistency)
con_consistency = np.ones(len(consistency_scores))
mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

print(f"MC Dropout without XAI Flagging:")
mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(mc_flags_uq_only, "FCNN MC Dropout only", config)

print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")


MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 16 Accuracy:62.5%
GREEN: Count: 184 Accuracy:89.1%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_mc_dropout_only.png
MC Dropout with XAI green accuracy: 0.8837
MC Dropout without XAI green accuracy: 0.8913
Improvement with XAI: -0.00759999999999994


In [12]:
# Deep Ensemble without XAI (constant consistency)
de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

print(f"Deep Ensemble without XAI Flagging:")
de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
plot_flag_distribution(de_flags_uq_only, "FCNN Deep Ensembles only", config)

print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")


Deep Ensemble without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 21 Accuracy:52.4%
GREEN: Count: 179 Accuracy:91.1%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_deep_ensembles_only.png
Deep Ensembles with XAI green accuracy: 0.9048
Deep Ensembles without XAI green accuracy: 0.9106
Improvement with XAI: -0.005799999999999916


### Save golden baseline report

In [13]:
fcnn_baseline = {
    'model': 'FCNN',
    'accuracy': round(fcnn_accuracy, 4),
    'classification_report': fcnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging_mc': mc_flag_results,
    'flagging_de': de_flag_results,
    'flagging_mc_only': mc_only_flag_results,
    'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
    'flagging_de_only': de_only_flag_results,
    'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config,'Diabetes - FCNN golden baseline', stage1_result = fcnn_baseline )
save_report(report_output, 'TC_03_FCNN_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
